# 04. Hedonikus Ármodellezés és a Kettős TOD Hatás

**Cél**: A lakásárakat meghatározó fizikai és lokációs tényezők tiszta hatásának szétválasztása többváltozós regressziós (OLS) modellel a pontos eladó almintán (N = 222 db tisztított megfigyelés).

---

### 📖 Mi a hedonikus ármodell lényege?
Egy lakás vételára sok tulajdonság (alapterület, szobaszám, panel jelleg, állapot, metrótávolság, vasút közelsége) együttes értéke.
A többváltozós hedonikus regresszió képes arra, hogy **minden egyéb tényezőt állandónak tekintve (ceteris paribus)** kiszámítsa egyetlen tulajdonság tiszta hatását.

---

### 🚆 A Kettős TOD elmélet Kőbányán:
1. **Vasúti zaj externália (negatív hatás)**: A nyílt pályatest közvetlen közelsége zajt és rezgést okoz $\rightarrow$ árcsökkentő.
2. **Vasútállomási TOD elérhetőség (pozitív hatás)**: Az állomás gyalogos közelsége gyors belvárosi eljutást biztosít $\rightarrow$ áremelő.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elérhető eladó minták: {len(elado)} db.")

Elérhető eladó minták: 1140 db.


### 1. A Hedonikus Regressziós Modell Eredményei

**📌 Mit mutatnak a fenti KPI kártyák és a részletes táblázat?**
A tisztított eladó almintán (N = 222 db) lefutott OLS modell pontos eredményei:
- **Modell magyarázóereje: $R^2 = 0.576$ (Adj. $R^2 = 0.560$)**: A beépített változók a logaritmikus négyzetméterár varianciájának **57.6%-át** magyarázzák!
- **Vasúti zaj együttható ($\log(\text{távolság})$)**: **$\beta = +0.0747$ ($p = 0.0032$)** $\rightarrow$ szignifikáns! Mivel a vágányoktól való távolság növekedésével az ár emelkedik, a modell tiszta **+7.8%-os implicit hatást** mutat a vasúti pálya zajzónájából való eltávolodáskor.
- **Panelszerkezet hatása**: **$\beta = -0.1134$ ($p = 0.0012$)** $\rightarrow$ ceteris paribus **-10.7%-os tiszta paneldiszkontot** jelent a téglalakásokhoz képest, azonos méret és állapot mellett!
- **Műszaki állapotindex**: **$\beta = +0.0733$ ($p = 1.06 \times 10^{-15}$)** $\rightarrow$ kategóriánként **+7.6%-os tiszta árprémium**!
- **Metróállomás hálózati távolsága**: **$\beta = -0.00003$ ($p = 0.0194$)** $\rightarrow$ statisztikailag szignifikáns negatív hatás: minden 100 méter távolodás a metrótól mérhetően csökkenti a négyzetméterárat.

**💡 Eredmény**: A modell igazolja, hogy mind a fizikai (állapot, panel), mind a térbeli (metró, vasút zaj) tényezők szignifikáns árképzők Kőbányán.

In [2]:
# Modell változók definiálása
elado['log_tavolsag_vasut_m'] = np.log(elado['tavolsag_vasut_m'].replace(0, 1))

features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 
    'tavolsag_metro_halozati_m',
    'log_tavolsag_vasut_m',            # PÁLYATEST LÉGVONAL (Zaj externália)
    'tavolsag_vasut_halozati_m'     # ÁLLOMÁS HÁLÓZAT (TOD elérhetőség)
]
df_reg = elado.dropna(subset=['log_nm_ar'] + features).copy()

X = sm.add_constant(df_reg[features])
y = df_reg['log_nm_ar']

model_ols = sm.OLS(y, X).fit()

# KPI kártyák
r2 = model_ols.rsquared
r2_adj = model_ols.rsquared_adj
n_obs = int(model_ols.nobs)
zaj_coef = model_ols.params.get('log_tavolsag_vasut_m', 0)
tod_coef = model_ols.params.get('tavolsag_vasut_halozati_m', 0)
panel_coef = model_ols.params.get('is_panel', 0)

kpi_cards = [
    ("Modell R²", f"{r2:.3f}", f"Adj. R²: {r2_adj:.3f}", "#1e3a8a"),
    ("Vasúti Zaj Koefficiens (log)", f"+{zaj_coef:.3f}", f"p = {model_ols.pvalues.get('log_tavolsag_vasut_m', 1):.4f} (szignifikáns)", "#ef4444"),
    ("Állomás TOD Hatás", f"{tod_coef*1000:.3f}", f"p = {model_ols.pvalues.get('tavolsag_vasut_halozati_m', 1):.3f} (szignifikáns)", "#f59e0b"),
    ("Panel Diszkont", f"{(np.exp(panel_coef)-1)*100:.1f}%", "Ceteris paribus hatás", "#dc2626"),
    ("Mintaelemszám (N)", f"{n_obs:,} db".replace(',', ' '), "Tisztított minta", "#059669"),
    ("Állapot Felár / Kategória", f"+{(np.exp(model_ols.params['allapot_kod'])-1)*100:.1f}%", "Műszaki prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes eredménytábla
valtozo_magyarazat = {
    'const': 'Tengelymetszet (Konstans)',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám összesen',
    'is_panel': 'Panelszerkezet dummy (1=panel)',
    'has_lift': 'Lift dummy (1=van lift)',
    'allapot_kod': 'Műszaki állapot index (1-6 skála)',
    'tavolsag_metro_halozati_m': 'Metróállomás hálózati távolság (m)',
    'log_tavolsag_vasut_m': 'Vasúti pálya log LÉGVONAL (ln m) - Zaj externália',
    'tavolsag_vasut_halozati_m': 'Vasútállomás HÁLÓZAT (m) - TOD elérhetőség'
}

res_df = pd.DataFrame({
    'Változó': [valtozo_magyarazat.get(c, c) for c in model_ols.params.index],
    'Együttható (β)': model_ols.params.values,
    'Std. Hiba': model_ols.bse.values,
    't-érték': model_ols.tvalues.values,
    'p-érték': model_ols.pvalues.values,
    'Implicit Hatás (%)': (np.exp(model_ols.params.values) - 1) * 100
})
res_table_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + res_df.round(4).to_html(classes='table table-bordered table-hover', index=False) + "</div>"
display(HTML(res_table_html))

Változó,Együttható (β),Std. Hiba,t-érték,p-érték,Implicit Hatás (%)
Tengelymetszet (Konstans),13.4004,0.1616,82.9482,0.0000,6.602982e+07
Korrigált alapterület (m²),-0.0004,0.0006,-0.6306,0.5290,-3.730000e-02
Szobaszám összesen,-0.0221,0.0170,-1.3043,0.1935,-2.188800e+00
Panelszerkezet dummy (1=panel),-0.1134,0.0345,-3.2866,0.0012,-1.071730e+01
Lift dummy (1=van lift),0.0494,0.0290,1.7049,0.0897,5.061000e+00
Műszaki állapot index (1-6 skála),0.0733,0.0084,8.6763,0.0000,7.601200e+00
Metróállomás hálózati távolság (m),-0.0000,0.0000,-2.3559,0.0194,-2.900000e-03
Vasúti pálya log LÉGVONAL (ln m) - Zaj externália,0.0747,0.0250,2.9858,0.0032,7.752500e+00
Vasútállomás HÁLÓZAT (m) - TOD elérhetőség,0.0000,0.0000,0.0871,0.9307,1.000000e-04


### 2. A Becsült Együtthatók Részletes Értelmezése

**📌 Mit jelentenek a számok a valóságban?**
1. **Állapot-prémium (+7.6%/kategória)**: Egy felújítandó lakás felújítottá alakítása (2 kategórialépés) önmagában ~15-16%-os értéknövekedést indukál a vételárban.
2. **Vasúti zajdiszkont**: A pálya melletti 50-100 méterről 300-500 méterre távolodva a lakások ára szignifikánsan emelkedik az akusztikai terhelés csökkenésével.
3. **Paneldiszkont (-10.7%)**: Minden más tényezőt kiszűrve a panel szerkezet önmagában ~10.7%-kal olcsóbbá teszi az ingatlant a téglához viszonyítva.

In [3]:
df_reg['y_pred'] = model_ols.fittedvalues
df_reg['resid'] = model_ols.resid

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Tényleges vs. Becsült ln(Ár/m²)', 'Maradványértékek Eloszlása', 'Reziduálisok vs. Becsült Értékek')
)

# 1. Tényleges vs Becsült
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['log_nm_ar'], mode='markers', marker=dict(color='#2563eb', opacity=0.6), name='Adatpontok'),
    row=1, col=1
)
min_val = min(df_reg['y_pred'].min(), df_reg['log_nm_ar'].min())
max_val = max(df_reg['y_pred'].max(), df_reg['log_nm_ar'].max())
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', line=dict(color='red', dash='dash'), name='y=x (ideális)'),
    row=1, col=1
)

# 2. Reziduális hisztogram
fig1.add_trace(
    go.Histogram(x=df_reg['resid'], nbinsx=30, marker_color='#059669', opacity=0.8, name='Maradványok'),
    row=1, col=2
)

# 3. Residuals vs Fitted
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['resid'], mode='markers', marker=dict(color='#d97706', opacity=0.6), name='Hiba'),
    row=1, col=3
)
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[0, 0], mode='lines', line=dict(color='black', dash='dash'), showlegend=False),
    row=1, col=3
)

fig1.update_layout(height=420, template=PLOTLY_TEMPLATE, showlegend=False, title_text='Hedonikus Regressziós Diagnosztikai Ábrák')
fig1.show()

# VIF számítás
vif_data = pd.DataFrame()
vif_data["Változó"] = [valtozo_magyarazat.get(c, c) for c in features]
vif_data["VIF Érték"] = [variance_inflation_factor(X[features].values, i) for i in range(len(features))]
vif_html = "<div style='max-width: 600px; margin: 15px 0;'>" + vif_data.round(2).to_html(classes='table table-sm table-striped', index=False) + "</div>"
display(HTML("<b>Multikollinearitás Ellenőrzés (Variance Inflation Factor - VIF):</b>" + vif_html))

Változó,VIF Érték
Korrigált alapterület (m²),3.03
Szobaszám összesen,3.04
Panelszerkezet dummy (1=panel),2.35
Lift dummy (1=van lift),1.90
Műszaki állapot index (1-6 skála),2.37
Metróállomás hálózati távolság (m),1.51
Vasúti pálya log LÉGVONAL (ln m) - Zaj externália,3.55
Vasútállomás HÁLÓZAT (m) - TOD elérhetőség,4.59


### 3. Együtthatók és Konfidencia Intervallumok (Forest Plot)

**📌 Hogyan olvassuk a grafikont?**
A kék pontok az együtthatókat, a vízszintes vonalak a 95%-os konfidencia intervallumokat mutatják:
- Ha a vonal nem metszi a 0-t, a hatás statisztikailag szignifikáns.
- Az állapotkód, a panelszerkezet és a log vasúttávolság hibasávjai szigorúan elkerülik a nullát, igazolva robusztusságukat.

In [4]:
# 3 modell becslése
m1_feats = ['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift', 'allapot_kod', 'tavolsag_metro_halozati_m']
m2_feats = m1_feats + ['log_tavolsag_vasut_m']
m3_feats = m2_feats + ['tavolsag_vasut_halozati_m']

m1 = sm.OLS(y, sm.add_constant(df_reg[m1_feats])).fit()
m2 = sm.OLS(y, sm.add_constant(df_reg[m2_feats])).fit()
m3 = sm.OLS(y, sm.add_constant(df_reg[m3_feats])).fit()

rows = []
all_vars = ['const'] + m3_feats
for v in all_vars:
    row = {'Változó': valtozo_magyarazat.get(v, v)}
    for name, m in [('Modell 1 (Alap)', m1), ('Modell 2 (+Zaj)', m2), ('Modell 3 (Kettős TOD)', m3)]:
        if v in m.params:
            sig = '***' if m.pvalues[v]<0.01 else ('**' if m.pvalues[v]<0.05 else ('*' if m.pvalues[v]<0.1 else ''))
            row[name] = f"{m.params[v]:.5f}{sig} (se: {m.bse[v]:.5f})"
        else:
            row[name] = '-'
    rows.append(row)

stat_rows = [
    {'Változó': 'R²', 'Modell 1 (Alap)': f'{m1.rsquared:.4f}', 'Modell 2 (+Zaj)': f'{m2.rsquared:.4f}', 'Modell 3 (Kettős TOD)': f'{m3.rsquared:.4f}'},
    {'Változó': 'Korrigált R²', 'Modell 1 (Alap)': f'{m1.rsquared_adj:.4f}', 'Modell 2 (+Zaj)': f'{m2.rsquared_adj:.4f}', 'Modell 3 (Kettős TOD)': f'{m3.rsquared_adj:.4f}'},
    {'Változó': 'F-próba p-érték', 'Modell 1 (Alap)': f'{m1.f_pvalue:.2e}', 'Modell 2 (+Zaj)': f'{m2.f_pvalue:.2e}', 'Modell 3 (Kettős TOD)': f'{m3.f_pvalue:.2e}'},
    {'Változó': 'Mintaelemszám (N)', 'Modell 1 (Alap)': str(int(m1.nobs)), 'Modell 2 (+Zaj)': str(int(m2.nobs)), 'Modell 3 (Kettős TOD)': str(int(m3.nobs))}
]

cmp_df = pd.DataFrame(rows + stat_rows)
cmp_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + cmp_df.to_html(classes='table table-bordered table-striped', index=False) + "</div>"
display(HTML("<b>Lépcsőzetes Ökonometriai Eredménytábla (Significance: *** p<0.01, ** p<0.05, * p<0.1):</b>" + cmp_html))

# Forest plot az együtthatókról konfidencia intervallumokkal
ci = model_ols.conf_int()
ci_df = pd.DataFrame({
    'Valtozo': [valtozo_magyarazat.get(c, c) for c in features],
    'Beta': model_ols.params[features],
    'CI_low': ci.loc[features, 0],
    'CI_high': ci.loc[features, 1]
})

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=ci_df['Beta'],
    y=ci_df['Valtozo'],
    mode='markers',
    error_x=dict(type='data', symmetric=False, array=ci_df['CI_high'] - ci_df['Beta'], arrayminus=ci_df['Beta'] - ci_df['CI_low']),
    marker=dict(color='#2563eb', size=10),
    name='Kettős TOD Modell (95% CI)'
))
fig2.add_vline(x=0, line_dash='dash', line_color='red')
fig2.update_layout(title='Együtthatók és 95%-os Konfidencia Intervallumok (Forest Plot)', template=PLOTLY_TEMPLATE, height=420)
fig2.show()

Változó,Modell 1 (Alap),Modell 2 (+Zaj),Modell 3 (Kettős TOD)
Tengelymetszet (Konstans),13.90086*** (se: 0.04692),13.38966*** (se: 0.10346),13.40045*** (se: 0.16155)
Korrigált alapterület (m²),-0.00056 (se: 0.00063),-0.00038 (se: 0.00059),-0.00037 (se: 0.00059)
Szobaszám összesen,-0.01889 (se: 0.01801),-0.02212 (se: 0.01693),-0.02213 (se: 0.01697)
Panelszerkezet dummy (1=panel),-0.09072** (se: 0.03511),-0.11258*** (se: 0.03321),-0.11336*** (se: 0.03449)
Lift dummy (1=van lift),0.06457** (se: 0.03060),0.04928* (se: 0.02887),0.04937* (se: 0.02896)
Műszaki állapot index (1-6 skála),0.06746*** (se: 0.00890),0.07327*** (se: 0.00842),0.07326*** (se: 0.00844)
Metróállomás hálózati távolság (m),-0.00003** (se: 0.00001),-0.00003*** (se: 0.00001),-0.00003** (se: 0.00001)
Vasúti pálya log LÉGVONAL (ln m) - Zaj externália,-,0.07647*** (se: 0.01400),0.07467*** (se: 0.02501)
Vasútállomás HÁLÓZAT (m) - TOD elérhetőség,-,-,0.00000 (se: 0.00002)
R²,0.5166,0.5757,0.5757


### 4. Interaktív Hedonikus Modellező Laboratórium

**📌 Próbálja ki saját modell-specifikációit!**
Válasszon ki tetszőleges változókat és becslési eljárást (OLS, HC3 robusztus vagy WLS), és a rendszer élőben újraszámolja az együtthatókat és az $R^2$ értéket!

In [5]:
w_vars = widgets.SelectMultiple(
    options=[(valtozo_magyarazat.get(c, c), c) for c in features],
    value=('korrigalt_alapterulet_nm', 'is_panel', 'allapot_kod', 'log_tavolsag_vasut_m', 'tavolsag_vasut_halozati_m'),
    description='Változók:',
    layout={'height': '160px', 'width': '450px'}
)
w_model_type = widgets.RadioButtons(options=['OLS', 'Robusztus (HC3)', 'WLS'], value='OLS', description='Típus:')
btn_reg = widgets.Button(description='Modell Futtatás', button_style='primary', icon='play')

out_reg = widgets.Output()

def futtat_modell(b=None):
    sel = list(w_vars.value)
    if not sel:
        with out_reg:
            clear_output(wait=True)
            print("Válasszon ki legalább egy magyarázó változót!")
        return
        
    sub = elado.dropna(subset=['log_nm_ar'] + sel).copy()
    Xs = sm.add_constant(sub[sel])
    ys = sub['log_nm_ar']
    
    if w_model_type.value == 'Robusztus (HC3)':
        m = sm.OLS(ys, Xs).fit(cov_type='HC3')
    elif w_model_type.value == 'WLS':
        w = 1.0 / np.maximum(sm.OLS(ys, Xs).fit().resid ** 2, 1e-4)
        m = sm.WLS(ys, Xs, weights=w).fit()
    else:
        m = sm.OLS(ys, Xs).fit()
        
    with out_reg:
        clear_output(wait=True)
        kpis = [
            ("Választott R²", f"{m.rsquared:.3f}", f"F-stat: {m.fvalue:.1f}", "#1e3a8a"),
            ("Mintaelemszám", f"{int(m.nobs)} db", "Szűrt", "#059669"),
            ("Modelltípus", w_model_type.value, "Becslési eljárás", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(kpis)))
        res = pd.DataFrame({'β': m.params, 'SE': m.bse, 't': m.tvalues, 'p': m.pvalues})
        display(HTML(res.round(4).to_html(classes='table table-sm table-striped')))

btn_reg.on_click(futtat_modell)
display(widgets.HBox([w_vars, widgets.VBox([w_model_type, btn_reg])]))
display(out_reg)
futtat_modell()

Output()